In [34]:
import cx_Oracle
import pandas as pd

# === Step 1: Connect as SYSDBA ===
sys_conn = cx_Oracle.connect("sys", "12345", "localhost:1521/XEPDB1", mode=cx_Oracle.SYSDBA)
sys_cursor = sys_conn.cursor()

# === Step 2: Connect as FYP user ===
conn = cx_Oracle.connect("hr", "hr123", "localhost:1521/XEPDB1")
cursor = conn.cursor()

# === Step 1: Drop & create sandBox table ===
try:
    cursor.execute("DROP TABLE sandBox_Households PURGE")
    print("Dropped existing sandBox table.")
except:
    print("No existing sandBox table.")

create_table_sql = """
CREATE TABLE sandBox_Households (
    id VARCHAR2(50),
    hhId VARCHAR2(100),
    province VARCHAR2(100),
    division VARCHAR2(100),
    district VARCHAR2(100),
    rooms VARCHAR2(50),
    constructionDuration VARCHAR2(100),
    wallsType VARCHAR2(100),
    roofsType VARCHAR2(100),
    DrinkingWater VARCHAR2(100),
    lightSource VARCHAR2(100),
    kitchenFuel VARCHAR2(100)
)
"""
cursor.execute(create_table_sql)
print("Created sandBox table.")

# === Step 2: Load csv & Insert ===
csv_file = "households_mapped.csv"  # update with correct file path
df = pd.read_csv(csv_file)

insert_sql = """
INSERT INTO sandBox_Households (
    id, hhId, province, division, district,
    rooms, constructionDuration, wallsType, roofsType,
    DrinkingWater, lightSource, kitchenFuel
) VALUES (
    :1, :2, :3, :4, :5,
    :6, :7, :8, :9,
    :10, :11, :12
)
"""

for _, row in df.iterrows():
    cursor.execute(insert_sql, (
        str(row['id']),
        str(row['hhId']),
        str(row['province']),
        str(row['division']),
        str(row['district']),
        str(row['rooms']),
        str(row['constructionDuration']),
        str(row['wallsType']),
        str(row['roofsType']),
        str(row['DrinkingWater']),
        str(row['lightSource']),
        str(row['kitchenFuel'])
    ))

conn.commit()
print(f"Inserted {len(df)} rows into sandBox table.")

# === Step 3: Verify ===
cursor.execute("SELECT table_name FROM user_tables")
print("Tables in schema:", [r[0] for r in cursor.fetchall()])

cursor.execute("SELECT COUNT(*) FROM sandBox_Households")
print("Total rows in sandBox:", cursor.fetchone()[0])

cursor.close()
conn.close()


Dropped existing sandBox table.
Created sandBox table.
Inserted 9996 rows into sandBox table.
Tables in schema: ['SANDBOX_HOUSEHOLDS']
Total rows in sandBox: 9996


In [25]:
import cx_Oracle
import pandas as pd

# === Step 1: Connect as SYSDBA ===
sys_conn = cx_Oracle.connect("sys", "12345", "localhost:1521/XEPDB1", mode=cx_Oracle.SYSDBA)
sys_cursor = sys_conn.cursor()



# Create user/schema (if not exists)
try:
    sys_cursor.execute("CREATE USER HR IDENTIFIED BY hr123")
    print("User HR created.")
except:
    print("User HR already exists.")

# Grant permissions
try:
    sys_cursor.execute("GRANT CONNECT, RESOURCE TO HR")
    sys_cursor.execute("ALTER USER HR QUOTA UNLIMITED ON USERS")
    print("Granted privileges to HR.")
except:
    print("Privileges may already exist.")

sys_conn.commit()
sys_cursor.close()
sys_conn.close()


# === Step 2: Connect as FYP user ===
conn = cx_Oracle.connect("hr", "hr123", "localhost:1521/XEPDB1")
cursor = conn.cursor()



# === Step 1: Drop & create sandBox table ===
try:
    cursor.execute("DROP TABLE sandBox_Households PURGE")
    print("Dropped existing sandBox table.")
except:
    print("No existing sandBox table.")

create_table_sql = """
CREATE TABLE sandBox_Households (
    id VARCHAR2(50),
    hhId VARCHAR2(100),
    province VARCHAR2(100),
    division VARCHAR2(100),
    district VARCHAR2(100),
    rooms NUMBER,
    constructionDuration NUMBER,
    wallsType NUMBER,
    roofsType NUMBER,
    DrinkingWater NUMBER,
    lightSource NUMBER,
    kitchenFuel NUMBER
)
"""
cursor.execute(create_table_sql)
print("Created sandBox table.")


# === Step 2: Load csv & Insert ===
csv_file = "Households.csv"  # update with correct file path
df = pd.read_csv(csv_file)

insert_sql = """
INSERT INTO sandBox_Households (
    id, hhId, province, division, district,
    rooms, constructionDuration, wallsType, roofsType,
    DrinkingWater, lightSource, kitchenFuel
) VALUES (
    :1, :2, :3, :4, :5,
    :6, :7, :8, :9,
    :10, :11, :12
)
"""

for _, row in df.iterrows():
    cursor.execute(insert_sql, (
        str(row['id']),
        str(row['hhId']),
        str(row['province']),
        str(row['division']),
        str(row['district']),
        int(row['rooms']),
        int(row['constructionDuration']),
        int(row['wallsType']),
        int(row['roofsType']),
        int(row['DrinkingWater']),
        int(row['lightSource']),
        int(row['kitchenFuel'])
    ))

conn.commit()
print(f"Inserted {len(df)} rows into sandBox table.")


# === Step 3: Verify ===
cursor.execute("SELECT table_name FROM user_tables")
print("Tables in schema:", [r[0] for r in cursor.fetchall()])

cursor.execute("SELECT COUNT(*) FROM sandBox_Households")
print("Total rows in sandBox:", cursor.fetchone()[0])

cursor.close()
conn.close()


User HR created.
Granted privileges to HR.
No existing sandBox table.
Created sandBox table.
Inserted 9996 rows into sandBox table.
Tables in schema: ['SANDBOX_HOUSEHOLDS']
Total rows in sandBox: 9996


In [27]:
import pandas as pd

# Load your CSV
df = pd.read_csv("DATA/Households.csv")

# === Mapping dictionaries (from your codebook) ===

rooms_map = {
    1: "One", 2: "Two", 3: "Three", 4: "Four", 5: "Five",
    6: "Six", 7: "Seven", 8: "Eight", 9: "Nine or More"
}

construction_map = {
    1: "Under Construction",
    2: "3 years or Less than 3 years",
    3: "4 to 9 years",
    4: "10 to 19 years",
    5: "20 to 49 years",
    6: "50 years or More"
}

walls_map = {
    1: "Baked Bricks/Blocks/Stones",
    2: "Unbaked Bricks/Mud",
    3: "Wood/Bamboo",
    4: "Plywood/Cardboard",
    5: "Pre-Fabric",
    6: "Others"
}

roofs_map = {
    1: "RCC/RBC",
    2: "Cement/Iron Sheet",
    3: "Garder/T-Iron",
    4: "Wood/Bamboo",
    5: "Pre-Fabric",
    6: "Others"
}

drinking_water_map = {
    1: "Tap Water",
    2: "Motor Pump/Hand Pump(Bore Hole)",
    3: "Protected Well",
    4: "Un-Protected Well",
    5: "Others",
    6: "Tap Water",
    7: "Motor Pump/Hand Pump(Bore Hole)",
    8: "Protected Well",
    9: "Un-Protected Well",
    10: "Bottled Water",
    11: "Spring",
    12: "Canal/River/Pond",
    13: "Filtration Plant",
    14: "Tanker/Water Bearer",
    15: "Others"
}

light_source_map = {
    1: "Electricity",
    2: "Solar Panel",
    3: "Kerosene Oil",
    4: "Gas Lamp",
    5: "Generator",
    6: "Bio Gas",
    7: "Others"
}

kitchen_fuel_map = {
    1: "Wood",
    2: "Sui Gas",
    3: "LPG/LNG(Cylinder)",
    4: "Kerosene Oil",
    5: "Electricity",
    6: "Bio Gas",
    7: "Dung Piles",
    8: "Others"
}

# === Apply mappings ===
df["rooms"] = df["rooms"].map(rooms_map)
df["constructionDuration"] = df["constructionDuration"].map(construction_map)
df["wallsType"] = df["wallsType"].map(walls_map)
df["roofsType"] = df["roofsType"].map(roofs_map)
df["DrinkingWater"] = df["DrinkingWater"].map(drinking_water_map)
df["lightSource"] = df["lightSource"].map(light_source_map)
df["kitchenFuel"] = df["kitchenFuel"].map(kitchen_fuel_map)

# Save to a new CSV
df.to_csv("households_mapped.csv", index=False)

print("✅ Mapping complete. File saved as households_mapped.csv")


✅ Mapping complete. File saved as households_mapped.csv


In [1]:


import pandas as pd
import sqlite3
import os

# ================== CONFIG ==================
CSV_FILE = "C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\Households.csv"          # Update if path is different
MAPPED_CSV = "C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\households_mapped.csv"       # Temporary file
SQLITE_DB = "fyp.db"                       # This will be your main DB
TABLE_NAME = "SANDBOX_HOUSEHOLDS"          # Uppercase to match your style

# ================== STEP 1: Map codes to text ==================
print("Loading and mapping data from CSV...")

df = pd.read_csv(CSV_FILE)

# Your mapping dictionaries (exactly as you had)
rooms_map = {
    1: "One", 2: "Two", 3: "Three", 4: "Four", 5: "Five",
    6: "Six", 7: "Seven", 8: "Eight", 9: "Nine or More"
}

construction_map = {
    1: "Under Construction",
    2: "3 years or Less than 3 years",
    3: "4 to 9 years",
    4: "10 to 19 years",
    5: "20 to 49 years",
    6: "50 years or More"
}

walls_map = {
    1: "Baked Bricks/Blocks/Stones",
    2: "Unbaked Bricks/Mud",
    3: "Wood/Bamboo",
    4: "Plywood/Cardboard",
    5: "Pre-Fabric",
    6: "Others"
}

roofs_map = {
    1: "RCC/RBC",
    2: "Cement/Iron Sheet",
    3: "Garder/T-Iron",
    4: "Wood/Bamboo",
    5: "Pre-Fabric",
    6: "Others"
}

drinking_water_map = {
    1: "Tap Water",
    2: "Motor Pump/Hand Pump(Bore Hole)",
    3: "Protected Well",
    4: "Un-Protected Well",
    5: "Others",
    6: "Tap Water",
    7: "Motor Pump/Hand Pump(Bore Hole)",
    8: "Protected Well",
    9: "Un-Protected Well",
    10: "Bottled Water",
    11: "Spring",
    12: "Canal/River/Pond",
    13: "Filtration Plant",
    14: "Tanker/Water Bearer",
    15: "Others"
}

light_source_map = {
    1: "Electricity",
    2: "Solar Panel",
    3: "Kerosene Oil",
    4: "Gas Lamp",
    5: "Generator",
    6: "Bio Gas",
    7: "Others"
}

kitchen_fuel_map = {
    1: "Wood",
    2: "Sui Gas",
    3: "LPG/LNG(Cylinder)",
    4: "Kerosene Oil",
    5: "Electricity",
    6: "Bio Gas",
    7: "Dung Piles",
    8: "Others"
}

# Apply mappings
df["rooms"] = df["rooms"].map(rooms_map)
df["constructionDuration"] = df["constructionDuration"].map(construction_map)
df["wallsType"] = df["wallsType"].map(walls_map)
df["roofsType"] = df["roofsType"].map(roofs_map)
df["DrinkingWater"] = df["DrinkingWater"].map(drinking_water_map)
df["lightSource"] = df["lightSource"].map(light_source_map)
df["kitchenFuel"] = df["kitchenFuel"].map(kitchen_fuel_map)

# Optional: Save mapped version
df.to_csv(MAPPED_CSV, index=False)
print(f"Mapping complete. {len(df)} rows ready.")

# ================== STEP 2: Create SQLite DB and Insert Data ==================
print(f"Creating SQLite database: {SQLITE_DB}")

# Connect to SQLite (creates file if not exists)
conn = sqlite3.connect(SQLITE_DB)
cur = conn.cursor()

# Drop table if exists
cur.execute(f'DROP TABLE IF EXISTS "{TABLE_NAME}"')
print("Old table dropped (if existed).")

# Create table
create_sql = f"""
CREATE TABLE "{TABLE_NAME}" (
    id TEXT,
    hhId TEXT,
    province TEXT,
    division TEXT,
    district TEXT,
    rooms TEXT,
    constructionDuration TEXT,
    wallsType TEXT,
    roofsType TEXT,
    DrinkingWater TEXT,
    lightSource TEXT,
    kitchenFuel TEXT
)
"""
cur.execute(create_sql)
print("Table created.")

# Insert all rows using pandas (fastest and cleanest)
df.to_sql(TABLE_NAME, conn, if_exists='append', index=False)
print(f"Inserted {len(df)} rows into {TABLE_NAME}.")

# Verify
cur.execute(f'SELECT COUNT(*) FROM "{TABLE_NAME}"')
count = cur.fetchone()[0]
print(f"Verification: {count} rows in table.")

cur.execute('SELECT name FROM sqlite_master WHERE type="table"')
print("All tables in DB:", [r[0] for r in cur.fetchall()])

conn.commit()
conn.close()

print(f"\n✅ SUCCESS! Your data is now in {SQLITE_DB}")
print("You can now run your Streamlit app using SQLite — no Oracle needed!")

Loading and mapping data from CSV...


OSError: [Errno 22] Invalid argument: 'C:\x0cyp_oracle_ai_assistant\x0cyp_oracle_ai_assistant\\DATA\\Households.csv'

In [3]:
import pandas as pd 



df = pd.read_csv(r'C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\rawData.csv')
df.head(10)

,PVID,PV,DSID,DS,BLK_DESC,HOUSEHOLD_UID,RELATION_W_HEAD
0,2,PUNJAB,60,MIANWALI DISTRICT,147020103,1F80AF9980FE4BD4AF1FD1A2DC645166,1
1,2,PUNJAB,60,MIANWALI DISTRICT,147020103,1F80AF9980FE4BD4AF1FD1A2DC645166,3
2,2,PUNJAB,60,MIANWALI DISTRICT,147020103,1F80AF9980FE4BD4AF1FD1A2DC645166,3
3,2,PUNJAB,60,MIANWALI DISTRICT,147020103,1F80AF9980FE4BD4AF1FD1A2DC645166,2
4,2,PUNJAB,60,MIANWALI DISTRICT,147020103,1F80AF9980FE4BD4AF1FD1A2DC645166,3
5,2,PUNJAB,60,MIANWALI DISTRICT,147020103,B0D395EBF61E4AB1967D3DD9ED5C867B,1
6,2,PUNJAB,60,MIANWALI DISTRICT,147020103,54555DCC62214030AFD9C9F0E393BE1A,1
7,2,PUNJAB,60,MIANWALI DISTRICT,147020103,54555DCC62214030AFD9C9F0E393BE1A,2
8,2,PUNJAB,60,MIANWALI DISTRICT,147020103,54555DCC62214030AFD9C9F0E393BE1A,3
9,2,PUNJAB,60,MIANWALI DISTRICT,147020103,002B34F873564DE5A0444E71823882CD,1


In [ ]:
import pandas as pd 



df = pd.read_csv(r'C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\migration.csv')
df.head(10)

,PVID,PV,DSID,DS,BITTHDSID,BIRTHDS,BIRTHPVID,BIRTHPV,BIRTHCOUNTRY
0,2,PUNJAB,60,MIANWALI DISTRICT,60.0,MIANWALI DISTRICT,2.0,PUNJAB,PAKISTAN
1,2,PUNJAB,60,MIANWALI DISTRICT,60.0,MIANWALI DISTRICT,2.0,PUNJAB,PAKISTAN
2,2,PUNJAB,60,MIANWALI DISTRICT,60.0,MIANWALI DISTRICT,2.0,PUNJAB,PAKISTAN
3,2,PUNJAB,60,MIANWALI DISTRICT,60.0,MIANWALI DISTRICT,2.0,PUNJAB,PAKISTAN
4,2,PUNJAB,60,MIANWALI DISTRICT,60.0,MIANWALI DISTRICT,2.0,PUNJAB,PAKISTAN
5,2,PUNJAB,60,MIANWALI DISTRICT,2.0,LAKKI MARWAT DISTRICT,1.0,KHYBER PAKHTUNKHWA,PAKISTAN
6,2,PUNJAB,60,MIANWALI DISTRICT,60.0,MIANWALI DISTRICT,2.0,PUNJAB,PAKISTAN
7,2,PUNJAB,60,MIANWALI DISTRICT,60.0,MIANWALI DISTRICT,2.0,PUNJAB,PAKISTAN
8,2,PUNJAB,60,MIANWALI DISTRICT,60.0,MIANWALI DISTRICT,2.0,PUNJAB,PAKISTAN
9,2,PUNJAB,60,MIANWALI DISTRICT,60.0,MIANWALI DISTRICT,2.0,PUNJAB,PAKISTAN


: 

In [1]:
import pandas as pd 



df = pd.read_csv(r'C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\households_mapped.csv')
df.head(10)

,id,hhId,province,division,district,rooms,constructionDuration,wallsType,roofsType,DrinkingWater,lightSource,kitchenFuel
0,AAAcCBAAJAAOoeHAAH,F8726375A76548C7AC668CF195588597,KHYBER PAKHTUNKHWA,HAZARA DIVISION,TORGHAR DISTRICT,One,20 to 49 years,Unbaked Bricks/Mud,Wood/Bamboo,Tap Water,Solar Panel,Wood
1,AAAcCBAAJAAOogyAAG,D43EF0118A49484AB2707F109C6E2239,KHYBER PAKHTUNKHWA,PESHAWAR DIVISION,CHARSADDA DISTRICT,Three,3 years or Less than 3 years,Baked Bricks/Blocks/Stones,RCC/RBC,Motor Pump/Hand Pump(Bore Hole),Electricity,LPG/LNG(Cylinder)
2,AAAcCBAAJAAOog9AAE,80C85E2B92A043289DC73D4ED186A1B9,KHYBER PAKHTUNKHWA,KOHAT DIVISION,HANGU DISTRICT,Five,4 to 9 years,Baked Bricks/Blocks/Stones,Garder/T-Iron,Tap Water,Solar Panel,Wood
3,AAAcCBAAJAAOoliAAV,2A6455E261CE4ACE84D489C27F1B594C,KHYBER PAKHTUNKHWA,KOHAT DIVISION,KURRAM DISTRICT,Three,10 to 19 years,Unbaked Bricks/Mud,Wood/Bamboo,Spring,Electricity,Wood
4,AAAcCBAAJAAOokfAAI,48F807C98BDA4733A22143C3E43E4230,KHYBER PAKHTUNKHWA,BANNU DIVISION,BANNU DISTRICT,Two,10 to 19 years,Unbaked Bricks/Mud,Wood/Bamboo,Motor Pump/Hand Pump(Bore Hole),Electricity,Wood
5,AAAcCBAAJAAOogVAAU,62B4CDB010954DF59F05502134E84449,KHYBER PAKHTUNKHWA,PESHAWAR DIVISION,NOWSHERA DISTRICT,Four,10 to 19 years,Baked Bricks/Blocks/Stones,RCC/RBC,Motor Pump/Hand Pump(Bore Hole),Electricity,Sui Gas
6,AAAcCBAAJAAOojoAAJ,38ACC46C15A846D0B3B91D178A7BE930,KHYBER PAKHTUNKHWA,MALAKAND DIVISION,BUNER DISTRICT,Seven,10 to 19 years,Unbaked Bricks/Mud,Wood/Bamboo,Protected Well,Electricity,Wood
7,AAAcCBAAJAAOokTAAX,AB5CBB1A509C422EADEB72570A0E2F1C,KHYBER PAKHTUNKHWA,MALAKAND DIVISION,SWAT DISTRICT,Three,4 to 9 years,Baked Bricks/Blocks/Stones,RCC/RBC,Tap Water,Electricity,Wood
8,AAAcCBAAJAAOohcAAZ,92E827E28F8644DB8F0F34978D08811B,KHYBER PAKHTUNKHWA,MALAKAND DIVISION,SWAT DISTRICT,Three,20 to 49 years,Wood/Bamboo,Wood/Bamboo,Spring,Electricity,Wood
9,AAAcCBAAJAAOoloAAD,FF5E0C1F26CF41E689DD5F66BBD59DA6,KHYBER PAKHTUNKHWA,HAZARA DIVISION,HARIPUR DISTRICT,Two,20 to 49 years,Baked Bricks/Blocks/Stones,RCC/RBC,Tap Water,Electricity,LPG/LNG(Cylinder)


In [2]:
import pandas as pd 



df = pd.read_csv(r'C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\Households.csv')
df.head(10)

,id,hhId,province,division,district,rooms,constructionDuration,wallsType,roofsType,DrinkingWater,lightSource,kitchenFuel
0,AAAcCBAAJAAOoeHAAH,F8726375A76548C7AC668CF195588597,KHYBER PAKHTUNKHWA,HAZARA DIVISION,TORGHAR DISTRICT,1,5,2,4,1,2,1
1,AAAcCBAAJAAOogyAAG,D43EF0118A49484AB2707F109C6E2239,KHYBER PAKHTUNKHWA,PESHAWAR DIVISION,CHARSADDA DISTRICT,3,2,1,1,2,1,3
2,AAAcCBAAJAAOog9AAE,80C85E2B92A043289DC73D4ED186A1B9,KHYBER PAKHTUNKHWA,KOHAT DIVISION,HANGU DISTRICT,5,3,1,3,6,2,1
3,AAAcCBAAJAAOoliAAV,2A6455E261CE4ACE84D489C27F1B594C,KHYBER PAKHTUNKHWA,KOHAT DIVISION,KURRAM DISTRICT,3,4,2,4,11,1,1
4,AAAcCBAAJAAOokfAAI,48F807C98BDA4733A22143C3E43E4230,KHYBER PAKHTUNKHWA,BANNU DIVISION,BANNU DISTRICT,2,4,2,4,2,1,1
5,AAAcCBAAJAAOogVAAU,62B4CDB010954DF59F05502134E84449,KHYBER PAKHTUNKHWA,PESHAWAR DIVISION,NOWSHERA DISTRICT,4,4,1,1,2,1,2
6,AAAcCBAAJAAOojoAAJ,38ACC46C15A846D0B3B91D178A7BE930,KHYBER PAKHTUNKHWA,MALAKAND DIVISION,BUNER DISTRICT,7,4,2,4,3,1,1
7,AAAcCBAAJAAOokTAAX,AB5CBB1A509C422EADEB72570A0E2F1C,KHYBER PAKHTUNKHWA,MALAKAND DIVISION,SWAT DISTRICT,3,3,1,1,6,1,1
8,AAAcCBAAJAAOohcAAZ,92E827E28F8644DB8F0F34978D08811B,KHYBER PAKHTUNKHWA,MALAKAND DIVISION,SWAT DISTRICT,3,5,3,4,11,1,1
9,AAAcCBAAJAAOoloAAD,FF5E0C1F26CF41E689DD5F66BBD59DA6,KHYBER PAKHTUNKHWA,HAZARA DIVISION,HARIPUR DISTRICT,2,5,1,1,6,1,3


In [8]:
import pandas as pd 



df = pd.read_csv(r'C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\IdDescription.csv')
df.head(10)

,POPULATION ATTRIBUTES,Unnamed: 1
0,MEMBER'S GENGER,NaN
1,1,Male
2,2,Female
3,3,Transgender
4,AGE IN YEARS,NaN
5,0 to 99,0 = Less than 1 year & 99 = 99 and Above years
6,MARITAL STATUS,NaN
7,1,Never Married
8,2,Married
9,3,Widower/Widow


In [4]:
import pandas as pd 



df = pd.read_csv(r'C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\Population.csv')
df.head(10)

,id,popId,hhId,province,division,district,gender,ageInYears,maritalStatus,education,usualActivity
0,AAAb9lAAEAAI93FAAF,b092a5d8-00d5-4770-abbf-973ed8dd303c,000246DF01B74B72AE4C0DF11084670D,KHYBER PAKHTUNKHWA,PESHAWAR DIVISION,NOWSHERA DISTRICT,1,3,1,NaN,NaN
1,AAAb9lAAEAAI93FAAG,3e292753-5011-43a1-b6a9-ec7024aaeef8,000246DF01B74B72AE4C0DF11084670D,KHYBER PAKHTUNKHWA,PESHAWAR DIVISION,NOWSHERA DISTRICT,1,33,2,10.0,1.0
2,AAAb9lAAEAAI93FAAH,34e55555-0f67-4b7c-95f7-8f7babc5fe79,000246DF01B74B72AE4C0DF11084670D,KHYBER PAKHTUNKHWA,PESHAWAR DIVISION,NOWSHERA DISTRICT,2,29,2,NaN,NaN
3,AAAb9lAAEAAI93FAAI,095286d4-527c-43be-801f-cd87c5c0fdea,000246DF01B74B72AE4C0DF11084670D,KHYBER PAKHTUNKHWA,PESHAWAR DIVISION,NOWSHERA DISTRICT,1,5,1,0.0,NaN
4,AAAb9lAACAAPc69AAL,dc06d288-dcbe-43e5-9d41-26348a9a253e,000929EBE962412E8E584BF5B21ADA30,SINDH,HYDERABAD DIVISION,BADIN DISTRICT,2,37,2,NaN,3.0
5,AAAb9lAACAAPc69AAM,38e94716-22e4-4187-89a9-a3d6bdd2d8bb,000929EBE962412E8E584BF5B21ADA30,SINDH,HYDERABAD DIVISION,BADIN DISTRICT,1,12,1,3.0,NaN
6,AAAb9lAACAAPc69AAN,5623192e-f5e8-451e-a3e1-79fd9572c8ca,000929EBE962412E8E584BF5B21ADA30,SINDH,HYDERABAD DIVISION,BADIN DISTRICT,1,27,1,9.0,1.0
7,AAAb9lAACAAPc+KAAG,f332c9ea-3bcb-432d-a5d2-b9c1330dce8c,000929EBE962412E8E584BF5B21ADA30,SINDH,HYDERABAD DIVISION,BADIN DISTRICT,1,67,2,NaN,1.0
8,AAAb9lAACAAPc+KAAH,6362cc7e-88a4-4859-a2b2-ecff5200d360,000929EBE962412E8E584BF5B21ADA30,SINDH,HYDERABAD DIVISION,BADIN DISTRICT,2,61,2,NaN,NaN
9,AAAb9lAACAAPc+KAAI,e91629c9-2478-42f3-8de2-98e7d1667f54,000929EBE962412E8E584BF5B21ADA30,SINDH,HYDERABAD DIVISION,BADIN DISTRICT,2,77,3,NaN,NaN


In [9]:
import sqlite3
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA")

FYP_DB = DATA_DIR / "FYP.db"
HR_DB  = DATA_DIR / "HR.db"

FYP_TABLES = [
    ("Households.csv", "households"),
    ("Population.csv", "population"),
    ("rawData.csv", "rawdata"),
]

HR_TABLES = [
    ("IdDescription.csv", "id_description"),
    ("households_mapped.csv", "households_mapped"),
    ("migration.csv", "migration"),
]

def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [
        c.strip()
         .replace(" ", "_")
         .replace("-", "_")
         .replace("/", "_")
         .replace("(", "")
         .replace(")", "")
        for c in df.columns
    ]
    return df

def load_csv(db_path: Path, csv_name: str, table: str):
    csv_path = DATA_DIR / csv_name
    print(f"📥 Loading {csv_path.name} -> {db_path.name}:{table}")

    # IdDescription.csv is “weird” (2 columns, mixed rows). Still loadable.
    df = pd.read_csv(csv_path, dtype=str)  # dtype=str avoids type headaches
    df = normalize_cols(df)

    with sqlite3.connect(db_path) as conn:
        conn.execute("PRAGMA journal_mode=WAL;")
        conn.execute("PRAGMA synchronous=NORMAL;")

        df.to_sql(table, conn, if_exists="replace", index=False)

def add_indexes():
    # helpful indexes for joins/filtering
    with sqlite3.connect(FYP_DB) as conn:
        conn.execute('CREATE INDEX IF NOT EXISTS idx_population_hhid ON population(hhId);')
        conn.execute('CREATE INDEX IF NOT EXISTS idx_households_hhid ON households(hhId);')

    with sqlite3.connect(HR_DB) as conn:
        conn.execute('CREATE INDEX IF NOT EXISTS idx_households_mapped_hhid ON households_mapped(hhId);')

def main():
    # build FYP.db
    for csv_name, table in FYP_TABLES:
        load_csv(FYP_DB, csv_name, table)

    # build HR.db
    for csv_name, table in HR_TABLES:
        load_csv(HR_DB, csv_name, table)

    add_indexes()
    print("✅ Done. Created:", FYP_DB, "and", HR_DB)

if __name__ == "__main__":
    main()

📥 Loading Households.csv -> FYP.db:households
📥 Loading Population.csv -> FYP.db:population
📥 Loading rawData.csv -> FYP.db:rawdata
📥 Loading IdDescription.csv -> HR.db:id_description
📥 Loading households_mapped.csv -> HR.db:households_mapped
📥 Loading migration.csv -> HR.db:migration
✅ Done. Created: C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\FYP.db and C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\HR.db


In [10]:
import sqlite3
from pathlib import Path

FYP_DB = Path(r"C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\FYP.db")

def run(conn, q):
    conn.execute(q)
    conn.commit()

with sqlite3.connect(FYP_DB) as conn:
    cur = conn.cursor()

    # ---------- HOUSEHOLDS: add label columns ----------
    run(conn, 'ALTER TABLE households ADD COLUMN rooms_label TEXT;')
    run(conn, 'ALTER TABLE households ADD COLUMN constructionDuration_label TEXT;')
    run(conn, 'ALTER TABLE households ADD COLUMN wallsType_label TEXT;')
    run(conn, 'ALTER TABLE households ADD COLUMN roofsType_label TEXT;')
    run(conn, 'ALTER TABLE households ADD COLUMN DrinkingWater_label TEXT;')
    run(conn, 'ALTER TABLE households ADD COLUMN lightSource_label TEXT;')
    run(conn, 'ALTER TABLE households ADD COLUMN kitchenFuel_label TEXT;')

    # rooms
    run(conn, """
    UPDATE households
    SET rooms_label = CASE CAST(rooms AS TEXT)
        WHEN '1' THEN 'One'
        WHEN '2' THEN 'Two'
        WHEN '3' THEN 'Three'
        WHEN '4' THEN 'Four'
        WHEN '5' THEN 'Five'
        WHEN '6' THEN 'Six'
        WHEN '7' THEN 'Seven'
        WHEN '8' THEN 'Eight'
        WHEN '9' THEN 'Nine or More'
        ELSE NULL
    END;
    """)

    # constructionDuration
    run(conn, """
    UPDATE households
    SET constructionDuration_label = CASE CAST(constructionDuration AS TEXT)
        WHEN '1' THEN 'Under Construction'
        WHEN '2' THEN '3 years or Less than 3 years'
        WHEN '3' THEN '4 to 9 years'
        WHEN '4' THEN '10 to 19 years'
        WHEN '5' THEN '20 to 49 years'
        WHEN '6' THEN '50 years or More'
        ELSE NULL
    END;
    """)

    # wallsType
    run(conn, """
    UPDATE households
    SET wallsType_label = CASE CAST(wallsType AS TEXT)
        WHEN '1' THEN 'Baked Bricks/Blocks/Stones'
        WHEN '2' THEN 'Unbaked Bricks/Mud'
        WHEN '3' THEN 'Wood/Bamboo'
        WHEN '4' THEN 'Plywood/Cardboard'
        WHEN '5' THEN 'Pre-Fabric'
        WHEN '6' THEN 'Others'
        ELSE NULL
    END;
    """)

    # roofsType
    run(conn, """
    UPDATE households
    SET roofsType_label = CASE CAST(roofsType AS TEXT)
        WHEN '1' THEN 'RCC/RBC'
        WHEN '2' THEN 'Cement/Iron Sheet'
        WHEN '3' THEN 'Garder/T-Iron'
        WHEN '4' THEN 'Wood/Bamboo'
        WHEN '5' THEN 'Pre-Fabric'
        WHEN '6' THEN 'Others'
        ELSE NULL
    END;
    """)

    # DrinkingWater (inside + outside mixed in your mapping)
    run(conn, """
    UPDATE households
    SET DrinkingWater_label = CASE CAST(DrinkingWater AS TEXT)
        WHEN '1' THEN 'Tap Water (Inside)'
        WHEN '2' THEN 'Motor Pump/Hand Pump(Bore Hole) (Inside)'
        WHEN '3' THEN 'Protected Well (Inside)'
        WHEN '4' THEN 'Un-Protected Well (Inside)'
        WHEN '5' THEN 'Others (Inside)'
        WHEN '6' THEN 'Tap Water (Outside)'
        WHEN '7' THEN 'Motor Pump/Hand Pump(Bore Hole) (Outside)'
        WHEN '8' THEN 'Protected Well (Outside)'
        WHEN '9' THEN 'Un-Protected Well (Outside)'
        WHEN '10' THEN 'Bottled Water (Outside)'
        WHEN '11' THEN 'Spring (Outside)'
        WHEN '12' THEN 'Canal/River/Pond (Outside)'
        WHEN '13' THEN 'Filtration Plant (Outside)'
        WHEN '14' THEN 'Tanker/Water Bearer (Outside)'
        WHEN '15' THEN 'Others (Outside)'
        ELSE NULL
    END;
    """)

    # lightSource
    run(conn, """
    UPDATE households
    SET lightSource_label = CASE CAST(lightSource AS TEXT)
        WHEN '1' THEN 'Electricity'
        WHEN '2' THEN 'Solar Panel'
        WHEN '3' THEN 'Kerosene Oil'
        WHEN '4' THEN 'Gas Lamp'
        WHEN '5' THEN 'Generator'
        WHEN '6' THEN 'Bio Gas'
        WHEN '7' THEN 'Others'
        ELSE NULL
    END;
    """)

    # kitchenFuel
    run(conn, """
    UPDATE households
    SET kitchenFuel_label = CASE CAST(kitchenFuel AS TEXT)
        WHEN '1' THEN 'Wood'
        WHEN '2' THEN 'Sui Gas'
        WHEN '3' THEN 'LPG/LNG(Cylinder)'
        WHEN '4' THEN 'Kerosene Oil'
        WHEN '5' THEN 'Electricity'
        WHEN '6' THEN 'Bio Gas'
        WHEN '7' THEN 'Dung Piles'
        WHEN '8' THEN 'Others'
        ELSE NULL
    END;
    """)

    # ---------- POPULATION: add label columns ----------
    run(conn, 'ALTER TABLE population ADD COLUMN gender_label TEXT;')
    run(conn, 'ALTER TABLE population ADD COLUMN maritalStatus_label TEXT;')
    run(conn, 'ALTER TABLE population ADD COLUMN education_label TEXT;')
    run(conn, 'ALTER TABLE population ADD COLUMN usualActivity_label TEXT;')

    # gender
    run(conn, """
    UPDATE population
    SET gender_label = CASE CAST(gender AS TEXT)
        WHEN '1' THEN 'Male'
        WHEN '2' THEN 'Female'
        WHEN '3' THEN 'Transgender'
        ELSE NULL
    END;
    """)

    # maritalStatus
    run(conn, """
    UPDATE population
    SET maritalStatus_label = CASE CAST(maritalStatus AS TEXT)
        WHEN '1' THEN 'Never Married'
        WHEN '2' THEN 'Married'
        WHEN '3' THEN 'Widower/Widow'
        WHEN '4' THEN 'Divorced'
        WHEN '5' THEN 'Seperation'
        ELSE NULL
    END;
    """)

    # education
    run(conn, """
    UPDATE population
    SET education_label = CASE CAST(education AS TEXT)
        WHEN '0' THEN 'Under 1'
        WHEN '1' THEN 'Class 1'
        WHEN '2' THEN 'Class 2'
        WHEN '3' THEN 'Class 3'
        WHEN '4' THEN 'Class 4'
        WHEN '5' THEN 'Class 5'
        WHEN '6' THEN 'Class 6'
        WHEN '7' THEN 'Class 7'
        WHEN '8' THEN 'Middle'
        WHEN '9' THEN 'Matric or Equivalent'
        WHEN '10' THEN 'Intermediate or Equivalent'
        WHEN '11' THEN 'Graduate (2 years)'
        WHEN '12' THEN 'Graduate (4 Years)'
        WHEN '13' THEN 'Master or Equivalent'
        WHEN '14' THEN 'M.Phil/Ph.D'
        WHEN '15' THEN 'Diploma/Certificate'
        WHEN '16' THEN 'Others'
        ELSE NULL
    END;
    """)

    # usualActivity
    run(conn, """
    UPDATE population
    SET usualActivity_label = CASE CAST(usualActivity AS TEXT)
        WHEN '1' THEN 'Employee(Job)'
        WHEN '2' THEN 'Self-Employed(Agriculture)'
        WHEN '3' THEN 'Self Employed(Non-Agriculture)'
        WHEN '4' THEN 'Employer'
        WHEN '5' THEN 'Unpaid family Helper(Agriculture)'
        WHEN '6' THEN 'Unpaid family Helper(Non-Agriculture)'
        WHEN '7' THEN 'Apprentance(Working as trainee "chootoo")'
        WHEN '8' THEN 'Not Working but Seeking Work'
        WHEN '9' THEN 'Not working and not seeking work'
        ELSE NULL
    END;
    """)

print("✅ Decoding complete. New *_label columns added to FYP.db")

✅ Decoding complete. New *_label columns added to FYP.db


In [11]:
import pandas as pd
import os

# Use raw string → safest on Windows
file = r"C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\16. CPI Data from 2001-2025 3.xlsx"

# Make sure output folder exists (very useful)
output_folder = r"fyp_oracle_ai_assistant\DATA"
os.makedirs(output_folder, exist_ok=True)

try:
    excel = pd.ExcelFile(file)
    print(f"Found {len(excel.sheet_names)} sheets:")
    print("─" * 50)

    for sheet in excel.sheet_names:
        df = pd.read_excel(file, sheet_name=sheet)
        
        # Clean filename - replace invalid characters
        safe_name = "".join(c for c in sheet if c.isalnum() or c in (' ', '-', '_')).strip()
        if not safe_name:
            safe_name = f"sheet_{sheet}"
            
        csv_path = os.path.join(output_folder, f"{safe_name}.csv")
        
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')  # utf-8-sig good for Excel
        
        print(f"Saved: {csv_path}   ({len(df):,} rows)")

    print("All sheets converted successfully! 🎉")

except FileNotFoundError:
    print(f"File not found: {file}")
    print("Please check the path is correct.")
except Exception as e:
    print("Error occurred:")
    print(e)

Found 3 sheets:
──────────────────────────────────────────────────
Saved: fyp_oracle_ai_assistant\DATA\Base Year 2000-01.csv   (364 rows)
Saved: fyp_oracle_ai_assistant\DATA\Base Year 2007-08.csv   (537 rows)
Saved: fyp_oracle_ai_assistant\DATA\Base Year 2015-16.csv   (436 rows)
All sheets converted successfully! 🎉


In [13]:
import pandas as pd

df = pd.read_csv(r"C:\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\fyp_oracle_ai_assistant\DATA\all_prices_powerbi_clean.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(10))
print("\nDate col candidates:", [c for c in df.columns if "date" in c.lower() or "month" in c.lower()])
print("\nNumeric col candidates:", df.select_dtypes(include="number").columns.tolist())

Shape: (16912, 13)
Columns: ['date', 'yearmonth', 'vegetable_price', 'unit', 'qim', 'fuel', 'rainfall', 'temperature', 'humidity', 'soiltemp', 'windspeed', 'city', 'commodity']
         date yearmonth  vegetable_price  unit         qim    fuel   rainfall  \
0  2013-01-01   2013-01            166.0  (KG)  104.552718  102.41   0.545161   
1  2013-02-01   2013-02            140.0  (KG)  105.312125  103.40   9.475000   
2  2013-03-01   2013-03            112.0  (KG)  107.042093  103.40   2.325806   
3  2013-04-01   2013-04            130.0  (KG)   93.911537  102.60   0.780000   
4  2013-05-01   2013-05            134.0  (KG)   90.647903   97.89   0.370968   
5  2013-06-01   2013-06            168.0  (KG)   89.929969  100.30   1.986667   
6  2013-07-01   2013-07            164.0  (KG)   88.231839  102.50   3.048387   
7  2013-08-01   2013-08            186.0  (KG)   86.468673  104.80  13.025806   
8  2013-09-01   2013-09            120.0  (KG)   90.423257  109.40   1.086667   
9  2013-10-01

In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

from dotenv import load_dotenv
from openai import OpenAI
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

load_dotenv()

# Model selection (set this in .env if you want)
# Example: OPENAI_FORECAST_MODEL=gpt-5
OPENAI_FORECAST_MODEL = os.getenv("OPENAI_FORECAST_MODEL", os.getenv("OPENAI_MODEL", "gpt-4o-mini"))
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

st.set_page_config(page_title="Forecasting", layout="wide")
st.title("📈 Forecasting Assistant (ARIMA / SARIMA / SARIMAX)")

# ---------------------------
# Helpers
# ---------------------------
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def profile_dataset(df: pd.DataFrame) -> dict:
    info = {}
    info["shape"] = df.shape
    info["columns"] = df.columns.tolist()

    # expected columns in your clean file
    info["date_col"] = "date" if "date" in df.columns else None
    info["city_col"] = "city" if "city" in df.columns else None
    info["commodity_col"] = "commodity" if "commodity" in df.columns else None
    info["y_col"] = "vegetable_price" if "vegetable_price" in df.columns else None

    if info["date_col"]:
        d = pd.to_datetime(df[info["date_col"]], errors="coerce")
        info["date_min"] = str(d.min().date()) if pd.notna(d.min()) else None
        info["date_max"] = str(d.max().date()) if pd.notna(d.max()) else None

    if info["city_col"]:
        cities = sorted(df[info["city_col"]].dropna().astype(str).unique().tolist())
        info["cities"] = cities
        info["has_country_series"] = any(c.lower() in ["pakistan", "national"] for c in cities)
    else:
        info["cities"] = []
        info["has_country_series"] = False

    if info["commodity_col"]:
        info["commodities"] = sorted(df[info["commodity_col"]].dropna().astype(str).unique().tolist())
    else:
        info["commodities"] = []

    # numeric candidates
    info["numeric_cols"] = df.select_dtypes(include="number").columns.tolist()

    return info

def init_chat():
    if "fc_chat" not in st.session_state:
        st.session_state.fc_chat = [
            {"role": "assistant", "content":
             "Upload your CSV ✅\n\n"
             "Then ask me anything — I can explain the dataset, suggest what you can do with it, "
             "and run forecasts like:\n"
             "- Forecast POTATOES in Lahore for next 12 months\n"
             "- Compare SARIMA vs SARIMAX (with fuel/weather)\n"
             "\n"
             "When you ask for a forecast, I’ll prepare the series, choose a model, run it, and plot it."}
        ]

def df_head_for_llm(df: pd.DataFrame, n=8) -> list:
    return df.head(n).to_dict(orient="records")

def safe_list(x, n=60):
    return x[:n] if len(x) > n else x

def prepare_panel_series(df, date_col, y_col, city=None, commodity=None, freq="MS"):
    d = df.copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d[y_col] = pd.to_numeric(d[y_col], errors="coerce")
    d = d.dropna(subset=[date_col, y_col])

    if city is not None and "city" in d.columns:
        d = d[d["city"].astype(str) == str(city)]
    if commodity is not None and "commodity" in d.columns:
        d = d[d["commodity"].astype(str) == str(commodity)]

    # aggregate duplicates per date
    agg = {y_col: "mean"}
    d = d.groupby(date_col, as_index=True).agg(agg).sort_index()

    # enforce regular monthly frequency
    ts = d[y_col].asfreq(freq)
    ts = ts.interpolate(limit_direction="both")
    return ts

def prepare_exog(df, date_col, exog_cols, city=None, commodity=None, freq="MS", index=None):
    d = df.copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d = d.dropna(subset=[date_col])

    if city is not None and "city" in d.columns:
        d = d[d["city"].astype(str) == str(city)]
    if commodity is not None and "commodity" in d.columns:
        d = d[d["commodity"].astype(str) == str(commodity)]

    for c in exog_cols:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    # aggregate duplicates
    d = d.groupby(date_col, as_index=True)[exog_cols].mean().sort_index()
    ex = d.asfreq(freq).interpolate(limit_direction="both")

    if index is not None:
        ex = ex.reindex(index).interpolate(limit_direction="both")
    return ex

def run_forecast(ts: pd.Series, model_type: str, horizon: int, m: int = 12, exog=None, exog_future=None, test_pct: int = 20):
    ts = ts.dropna()
    n = len(ts)
    n_test = max(1, int(n * test_pct / 100))
    train, test = ts.iloc[:-n_test], ts.iloc[-n_test:]

    if exog is not None:
        exog_train = exog.iloc[:-n_test]
        exog_test = exog.iloc[-n_test:]
    else:
        exog_train = exog_test = None

    order = (1, 1, 1)
    seasonal_order = (0, 0, 0, 0) if model_type == "ARIMA" else (1, 1, 1, m)

    model = SARIMAX(
        train,
        exog=exog_train,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    res = model.fit(disp=False)

    pred_test = res.get_forecast(steps=len(test), exog=exog_test).summary_frame()
    yhat_test = pred_test["mean"].values

    metrics = {
        "MAE": float(mean_absolute_error(test.values, yhat_test)),
        "RMSE": rmse(test.values, yhat_test),
        "order": order,
        "seasonal_order": seasonal_order,
        "n_points": int(n)
    }

    fc = res.get_forecast(steps=int(horizon), exog=exog_future).summary_frame()
    return res, train, test, yhat_test, fc, metrics

# ---------------------------
# Upload
# ---------------------------
uploaded = st.file_uploader("Upload forecasting CSV", type=["csv"])

if uploaded:
    df = pd.read_csv(uploaded)
    st.session_state.fc_df = df
    st.session_state.fc_profile = profile_dataset(df)
    st.success("✅ File uploaded and loaded into session.")

df = st.session_state.get("fc_df", None)
profile = st.session_state.get("fc_profile", None)

# ---------------------------
# Dataset summary + controls
# ---------------------------
if df is not None and profile is not None:
    st.subheader("📌 Dataset Summary")
    c1, c2, c3, c4 = st.columns(4)
    c1.write(f"**Rows, Cols:** {profile['shape']}")
    c2.write(f"**Date range:** {profile.get('date_min')} → {profile.get('date_max')}")
    c3.write(f"**Commodities:** {len(profile.get('commodities', []))}")
    c4.write(f"**Areas:** {len(profile.get('cities', []))}")

    with st.expander("Show available commodities"):
        st.write(safe_list(profile.get("commodities", []), 500))

    with st.expander("Show available areas (city/country)"):
        st.write(safe_list(profile.get("cities", []), 500))

    st.subheader("Preview")
    st.dataframe(df.head(30), use_container_width=True)

st.divider()

# ---------------------------
# Forecast Command Panel (right side)
# ---------------------------
st.subheader("🧠 Forecast Command Panel + Chat")

init_chat()

left, right = st.columns([1.2, 1])

with right:
    st.markdown("### ⚙️ Forecast Settings")
    if df is None:
        st.info("Upload a CSV first.")
    else:
        date_col = profile.get("date_col") or st.selectbox("Date column", df.columns.tolist())
        y_col = profile.get("y_col") or st.selectbox("Target (y)", df.columns.tolist())

        city = None
        commodity = None

        if "city" in df.columns:
            city = st.selectbox("Area (city/country)", sorted(df["city"].dropna().astype(str).unique()))
        if "commodity" in df.columns:
            commodity = st.selectbox("Commodity", sorted(df["commodity"].dropna().astype(str).unique()))

        horizon = st.number_input("Forecast horizon (months)", 1, 120, 12)
        test_pct = st.slider("Test size (%)", 5, 40, 20)

        model_type = st.selectbox("Model type", ["SARIMA", "SARIMAX", "ARIMA"], index=0)
        m = st.number_input("Seasonality m (monthly=12)", 2, 60, 12)

        exog_candidates = ["qim","fuel","rainfall","temperature","humidity","soiltemp","windspeed"]
        exog_cols = []
        if model_type == "SARIMAX":
            exog_cols = st.multiselect("Exogenous variables", [c for c in exog_candidates if c in df.columns],
                                       default=["fuel","temperature"])

        run_btn = st.button("🚀 Run Forecast Now")

with left:
    # ---------------------------
    # Chat UI (real LLM)
    # ---------------------------
    for mmsg in st.session_state.fc_chat:
        with st.chat_message(mmsg["role"]):
            st.write(mmsg["content"])

    user_msg = st.chat_input("Ask anything (GK + forecasting). Example: Forecast POTATOES in Lahore next 12 months.")
    if user_msg:
        st.session_state.fc_chat.append({"role": "user", "content": user_msg})
        with st.chat_message("user"):
            st.write(user_msg)

        # Build context for the LLM (small + useful)
        dataset_context = ""
        if df is not None and profile is not None:
            dataset_context = f"""
            DATASET CONTEXT:
            - shape: {profile['shape']}
            - columns: {profile['columns']}
            - date range: {profile.get('date_min')} to {profile.get('date_max')}
            - sample rows (head): {df_head_for_llm(df, n=6)}
            - cities sample: {safe_list(profile.get('cities', []), 40)}
            - commodities sample: {safe_list(profile.get('commodities', []), 40)}
            Important columns:
            - date: {profile.get('date_col')}
            - target: {profile.get('y_col')}
            - segment columns: city, commodity
            - possible exogenous vars: qim, fuel, rainfall, temperature, humidity, soiltemp, windspeed
            """

        SYSTEM = (
            "You are ChatGPT: a helpful general knowledge assistant AND a professor of time series forecasting.\n"
            "You can answer general questions, but you also must help the user use their uploaded dataset.\n"
            "When the user asks for a forecast, you must respond with:\n"
            "1) A friendly explanation\n"
            "2) A JSON action plan (strict JSON) in a fenced block with keys:\n"
            "   {\"intent\":\"forecast|analysis|general\",\"city\":...,\"commodity\":...,\"horizon\":...,"
            "    \"model\":\"ARIMA|SARIMA|SARIMAX\",\"m\":12,"
            "    \"use_exog\":true|false,\"exog_cols\":[...]} \n"
            "If dataset is not uploaded, ask user to upload it.\n"
            "Prefer SARIMA for monthly series with seasonality m=12.\n"
            "Prefer SARIMAX if user asks to include drivers (fuel/weather) or accuracy.\n"
        )

        messages = [{"role": "system", "content": SYSTEM}]
        if dataset_context:
            messages.append({"role": "system", "content": dataset_context})

        # include last few messages to keep context short
        history_tail = st.session_state.fc_chat[-8:]
        for h in history_tail:
            messages.append({"role": h["role"], "content": h["content"]})

        with st.spinner("Thinking..."):
            resp = client.chat.completions.create(
                model=OPENAI_FORECAST_MODEL,
                messages=messages,
                temperature=0.3
            )
            assistant_text = resp.choices[0].message.content.strip()

        st.session_state.fc_chat.append({"role": "assistant", "content": assistant_text})
        with st.chat_message("assistant"):
            st.write(assistant_text)

        # Try to extract JSON action plan if present
        plan = None
        m = re.search(r"```json\s*(\{.*?\})\s*```", assistant_text, flags=re.DOTALL)
        if m:
            try:
                plan = json.loads(m.group(1))
            except Exception:
                plan = None

        # If it's a forecast intent, store plan so user can run it
        if plan and plan.get("intent") == "forecast":
            st.session_state.fc_plan = plan
            st.info("✅ Forecast plan detected. You can click 'Run Forecast Now' in the right panel.")

# ---------------------------
# Run forecast (from panel button OR from plan)
# ---------------------------
if df is not None and profile is not None:
    if run_btn:
        # Prefer LLM plan if exists; else use panel inputs
        plan = st.session_state.get("fc_plan", None)

        date_col_use = profile.get("date_col") or "date"
        y_col_use = profile.get("y_col") or "vegetable_price"

        city_use = city
        commodity_use = commodity
        horizon_use = int(horizon)
        model_use = model_type
        m_use = int(m)
        exog_use = exog_cols

        if plan:
            city_use = plan.get("city") or city_use
            commodity_use = plan.get("commodity") or commodity_use
            horizon_use = int(plan.get("horizon") or horizon_use)
            model_use = (plan.get("model") or model_use).upper()
            m_use = int(plan.get("m") or m_use)
            exog_use = plan.get("exog_cols") or exog_use

            if model_use not in ["ARIMA", "SARIMA", "SARIMAX"]:
                model_use = "SARIMA"

        # Prepare series
        ts = prepare_panel_series(df, date_col_use, y_col_use, city=city_use, commodity=commodity_use, freq="MS")

        # Prepare exog if SARIMAX
        exog = exog_future = None
        if model_use == "SARIMAX" and exog_use:
            exog = prepare_exog(df, date_col_use, exog_use, city=city_use, commodity=commodity_use, freq="MS", index=ts.index)
            # baseline future exog = last observed values repeated
            last = exog.iloc[-1]
            exog_future = pd.DataFrame([last.values] * horizon_use, columns=exog_use)

        st.subheader("📊 Forecast Result")
        st.write(f"**City/Area:** {city_use} | **Commodity:** {commodity_use} | **Model:** {model_use} | **Horizon:** {horizon_use} months")

        st.plotly_chart(px.line(ts, title="Historical Series"), use_container_width=True)

        # Run model
        try:
            res, train, test, yhat_test, fc, metrics = run_forecast(
                ts,
                model_type=("ARIMA" if model_use == "ARIMA" else ("SARIMA" if model_use == "SARIMA" else "SARIMAX")),
                horizon=horizon_use,
                m=m_use,
                exog=exog,
                exog_future=exog_future,
                test_pct=int(test_pct)
            )

            st.write("**Backtest metrics:**", metrics)

            fc_index = pd.date_range(start=ts.index[-1], periods=horizon_use + 1, freq="MS")[1:]
            fc = fc.copy()
            fc["date"] = fc_index

            # Plot forecast
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=train.index, y=train.values, name="Train"))
            fig.add_trace(go.Scatter(x=test.index, y=test.values, name="Test"))
            fig.add_trace(go.Scatter(x=test.index, y=yhat_test, name="Pred(Test)"))

            fig.add_trace(go.Scatter(x=fc["date"], y=fc["mean"], name="Forecast", line=dict(color="green")))
            fig.add_trace(go.Scatter(x=fc["date"], y=fc["mean_ci_lower"], name="Lower CI", line=dict(color="rgba(0,128,0,0.25)")))
            fig.add_trace(go.Scatter(
                x=fc["date"],
                y=fc["mean_ci_upper"],
                name="Upper CI",
                line=dict(color="rgba(0,128,0,0.25)")
            ))

            fig.update_layout(
                template="plotly_white",
                title="Forecast + Confidence Interval",
                xaxis_title="Date",
                yaxis_title=y_col_use
            )
            st.plotly_chart(fig, use_container_width=True)

            # ---- Quick "instant answer" (next forecast value) ----
            next_date = fc["date"].iloc[0]
            next_value = float(fc["mean"].iloc[0])
            st.success(f"✅ Expected next value for **{commodity_use}** in **{city_use}** on **{next_date.date()}** is approximately **{next_value:.2f}**")

            # ---- Forecast table ----
            out = fc[["date", "mean", "mean_ci_lower", "mean_ci_upper"]].copy()
            out.rename(columns={
                "mean": "forecast",
                "mean_ci_lower": "lower_ci",
                "mean_ci_upper": "upper_ci"
            }, inplace=True)

            st.subheader("📋 Forecast Values")
            st.dataframe(out, use_container_width=True)

            # ---- Download forecast ----
            csv_bytes = out.to_csv(index=False).encode("utf-8")
            st.download_button(
                "⬇️ Download forecast as CSV",
                data=csv_bytes,
                file_name=f"forecast_{str(city_use).replace(' ','_')}_{str(commodity_use).replace(' ','_')}.csv",
                mime="text/csv"
            )

            # ---- Optional: write to chat summary ----
            summary_text = (
                f"Forecast completed ✅\n"
                f"- City/Area: {city_use}\n"
                f"- Commodity: {commodity_use}\n"
                f"- Model: {model_use} (m={m_use})\n"
                f"- Horizon: {horizon_use} months\n"
                f"- Next forecast: {next_date.date()} → {next_value:.2f}\n"
                f"- Backtest MAE: {metrics.get('MAE'):.3f}, RMSE: {metrics.get('RMSE'):.3f}\n"
            )

            st.session_state.fc_chat.append({"role": "assistant", "content": summary_text})

        except Exception as e:
            st.error(f"❌ Forecast failed: {e}")
            st.session_state.fc_chat.append({"role": "assistant", "content": f"❌ Forecast failed: {e}"})

In [ ]:
from openai import OpenAI

# 🔴 PASTE YOUR API KEY HERE
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

try:
    response = client.responses.create(
        model="gpt-4.1-mini",
        input="Reply with only the word: OK"
    )

    print("API is WORKING ✅")
    print("Response:", response.output_text)

except Exception as e:
    print("API is NOT working ❌")
    print("Error:", e)


API is NOT working ❌
Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


In [ ]:
import requests
import json
import base64
from pathlib import Path

# ─── CONFIG ────────────────────────────────────────────────────────────────
PROXY_URL = "https://api.puter.com/v2/openai/v1/chat/completions"   # current endpoint from tutorial
MODEL = "gpt-4o"           # also works: gpt-4o-mini, o1-preview, o1-mini

headers = {
    "Content-Type": "application/json",
    # You don't need any api-key header anymore with this proxy
}

# ─── 1. Simple text prompt test ────────────────────────────────────────────
def test_text_prompt(prompt):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.7,
        "max_tokens": 500
    }

    try:
        r = requests.post(PROXY_URL, headers=headers, json=payload, timeout=90)
        r.raise_for_status()
        result = r.json()
        return result["choices"][0]["message"]["content"]
    except Exception as e:
        return f"Error: {str(e)}"

# ─── 2. Vision (image understanding) test ──────────────────────────────────
def test_vision(image_path: str | Path, prompt="Describe this image in detail"):
    # Read and base64 encode image
    with open(image_path, "rb") as f:
        img_data = base64.b64encode(f.read()).decode("utf-8")

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{img_data}"}
                    }
                ]
            }
        ],
        "max_tokens": 500
    }

    try:
        r = requests.post(PROXY_URL, headers=headers, json=payload, timeout=120)
        r.raise_for_status()
        result = r.json()
        return result["choices"][0]["message"]["content"]
    except Exception as e:
        return f"Vision error: {str(e)}"

# ─── 3. Quick test ─────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=== Text test ===\n")
    print(test_text_prompt("Tell me a very short joke about programmers"))

    print("\n\n=== Vision test (put a photo.jpg in the same folder) ===\n")
    try:
        print(test_vision("photo.jpg", "Describe this photo and tell me what mood it gives"))
    except FileNotFoundError:
        print("Put any photo.jpg in the folder to test vision")

In [19]:
import requests
import json
import time

# ─── Puter Proxy Configuration ─────────────────────────────────────────────
BASE_URL = "https://api.puter.com/v2/openai/v1/chat/completions"
HEADERS = {
    "Content-Type": "application/json",
    # No real API key needed — Puter proxy ignores it
}

PROMPT = "Write a short poem about coding"

MODELS_TO_TEST = [
    "gpt-5.2",
    "gpt-5.2-chat",
    "gpt-5.2-pro",          # usually routes via openrouter
    "gpt-5.1"
]

# ─── Function to call the proxy ────────────────────────────────────────────
def call_puter_model(model_name, prompt=PROMPT):
    print(f"\n{'═'*70}")
    print(f" Testing model: {model_name}")
    print(f"{'─'*70}")

    payload = {
        "model": model_name,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.7,
        "max_tokens": 180
    }

    try:
        response = requests.post(
            BASE_URL,
            headers=HEADERS,
            json=payload,
            timeout=60
        )
        response.raise_for_status()

        result = response.json()
        content = result["choices"][0]["message"]["content"].strip()

        print(content)
        print("\nSuccess ✓")

    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e}")
        if response.status_code == 429:
            print("→ Rate limit hit (too many requests)")
        elif response.status_code == 404:
            print("→ Model not found / disabled")
        else:
            print(f"Status code: {response.status_code}")
            print(response.text[:300] + "..." if len(response.text) > 300 else response.text)

    except Exception as e:
        print(f"Error: {str(e)}")

    # Small delay to be nice to the proxy
    time.sleep(1.5)


# ─── Run all tests ─────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("Starting Puter proxy model tests...")
    print(f"Prompt: {PROMPT}\n")

    for model in MODELS_TO_TEST:
        call_puter_model(model)

    print("\n" + "═"*70)
    print("All tests completed!")
    print("Note: These models are unofficial proxies → may stop working or be rate-limited.")

Starting Puter proxy model tests...
Prompt: Write a short poem about coding


══════════════════════════════════════════════════════════════════════
 Testing model: gpt-5.2
──────────────────────────────────────────────────────────────────────
HTTP Error: 403 Client Error: Forbidden for url: https://api.puter.com/v2/openai/v1/chat/completions
Status code: 403
Forbidden

══════════════════════════════════════════════════════════════════════
 Testing model: gpt-5.2-chat
──────────────────────────────────────────────────────────────────────
HTTP Error: 403 Client Error: Forbidden for url: https://api.puter.com/v2/openai/v1/chat/completions
Status code: 403
Forbidden

══════════════════════════════════════════════════════════════════════
 Testing model: gpt-5.2-pro
──────────────────────────────────────────────────────────────────────
HTTP Error: 403 Client Error: Forbidden for url: https://api.puter.com/v2/openai/v1/chat/completions
Status code: 403
Forbidden

════════════════════════════